In [16]:
from __future__ import annotations
from dataclasses import dataclass
from typing import Sequence
import numpy as np

In [44]:

ATOMIC_NUMBER = {"H": 1, "C": 6}

@dataclass     # iinit funktion wird damit schon definiert
class Atom:
    symbol: str
    coord: Sequence[float]

    def __post_init__(self) -> None:   #liefert nichts zurück
        self.symbol = self.symbol.strip().capitalize()
        self.coord = np.asarray(self.coord, dtype = float)

    def __str__(self):   #helferfuntionenn dunder-Funktion __name__
        return f"{self.symbol} {self.coord[0]:.6f} {self.coord[1]:.6f} {self.coord[2]:.6f}"
     
    @property
    def atomic_number(self) -> int:
        return ATOMIC_NUMBER[self.symbol]
    
    @property
    def n_electrons(self) -> int:
        return self.atomic_number
    


    @classmethod
    def from_string(cls, s: str) -> Atom:
        parts = s.split()   # spaltet string in liste von substrings
        if len(parts) != 4:
            raise ValueError(f"Expected 'Symbol X Y Z', got {s}")
        symbol = parts[0]   # 0. element der liste
        try:
            coord = np.asarray(parts[1:], dtype=float)
        except ValueError as exc:
            raise ValueError(f"Invalid coordinates in atom string {s}") from exc
        return cls(symbol, coord)

    def get_distance(self, other: Atom) -> float:
        if not isinstance(other, Atom): 
            raise ValueError(f"Distance can only be calculated between two Atom intstances")
        return float(np.linalg.norm(self.coord - other.coord))


In [45]:
hatom1 = Atom('h', [0.0, 1.0, 2.0])
hatom2 = Atom('H', [0.0, 0.0, 2.0])
print(hatom1)
print(hatom2)

H 0.000000 1.000000 2.000000
H 0.000000 0.000000 2.000000


In [46]:
s = "C 0.0 0.0 0.0"
c_atom = Atom.from_string(s)
h_atom = Atom("h", np.zeros(3))
h_atom.__str__()


'H 0.000000 0.000000 0.000000'

In [47]:
atom1 = Atom("C", np.zeros(3))
atom2 = Atom("C", np.ones(3))
atom1.get_distance(atom2)
print(atom2.atomic_number)
print(atom2.n_electrons)

6
6


In [61]:
@dataclass
class Molecule:
    atoms: List[Atom]

    @classmethod
    def from_string(cls, xyz_string: str) -> Molecule:
        lines = xyz_string.strip().splitlines()
        atoms = []
        for line in lines[2:]:
            atom = Atom.from_string(line)
            atoms.append(atom)
        return cls(atoms)
    
    def __str__(self):
        return "\n".join([str(atom) for atom in self.atoms])   # string abc das jeden wert in neue zeile schreibt

    @property
    def n_electrons(self):
       return sum([atom.n_electrons for atom in self.atoms])   # list comprehension
    
    def to_string(self) -> str:   # hier wird die xyz string generiert, die dann für die visualisierung genutzt werden kann
        lines = [str(len(self.atoms)), "Generated by Molecule Class"] 
        for atom in self.atoms:
            line = f"{atom.symbol} {atom.coord[0]:.6f} {atom.coord[1]:.6f} {atom.coord[2]:.6f}"
            lines.append(line)
        return "\n".join(lines) # string abc das jeden wert in neue zeile schreibt  

        
 

In [55]:
atomlist = [atom1, atom2]
mol = Molecule(atomlist)

In [62]:
benzene_xyz = """12
Benzene molecule
C 0.000000 1.402720 0.000000
C 1.214790 0.701360 0.000000
C 1.214790 -0.701360 0.000000
C 0.000000 -1.402720 0.000000
C -1.214790 -0.701360 0.000000
C -1.214790 0.701360 0.000000
H 0.000000 2.490290 0.000000
H 2.156660 1.245150 0.000000
H 2.156660 -1.245150 0.000000
H 0.000000 -2.490290 0.000000
H -2.156660 -1.245150 0.000000
H -2.156660 1.245150 0.000000
"""
benzene = Molecule.from_string(benzene_xyz)
print(benzene)
print(benzene.to_string())

C 0.000000 1.402720 0.000000
C 1.214790 0.701360 0.000000
C 1.214790 -0.701360 0.000000
C 0.000000 -1.402720 0.000000
C -1.214790 -0.701360 0.000000
C -1.214790 0.701360 0.000000
H 0.000000 2.490290 0.000000
H 2.156660 1.245150 0.000000
H 2.156660 -1.245150 0.000000
H 0.000000 -2.490290 0.000000
H -2.156660 -1.245150 0.000000
H -2.156660 1.245150 0.000000
12
Generated by Molecule Class
C 0.000000 1.402720 0.000000
C 1.214790 0.701360 0.000000
C 1.214790 -0.701360 0.000000
C 0.000000 -1.402720 0.000000
C -1.214790 -0.701360 0.000000
C -1.214790 0.701360 0.000000
H 0.000000 2.490290 0.000000
H 2.156660 1.245150 0.000000
H 2.156660 -1.245150 0.000000
H 0.000000 -2.490290 0.000000
H -2.156660 -1.245150 0.000000
H -2.156660 1.245150 0.000000


In [76]:
import py3Dmol

def visualize(molecule: Molecule):
    xyz = molecule.to_string()  
    view = py3Dmol.view(width=400, height=400)
    view.addModel(xyz, "xyz") # hier muss noch die xyz string übergeben werden, die aus molecule generiert werden muss
    view.setStyle({'stick': {}})
    view.zoomTo()
    return view


In [77]:
view = visualize(benzene)
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [81]:
import numpy as np
from dataclasses import field
@dataclass
class Shell:
    l : int # 0 s, 1 p, 2 d, 3 f 
    exponents: np.ndarray # 1d array of exponents
    coefficients: np.ndarray # 1d array of coefficients
    norm_factors : np.ndarray = field(init=False) # wird nicht im init constructor übergeben, sondern wird automatisch berechnet
    center: np.ndarray = field(init=False)



In [82]:
shell = Shell(0, np.ones(10), np.ones(10))

In [83]:
print(shell.exponents)

[1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [85]:
@dataclass  # basissatz aus dictonary aus shells
class BasisSet:
    name: str
    elements: dict[str, list[Shell]] = field(default_factory=dict) # default_factory ist eine Funktion, die aufgerufen wird, um den Standardwert für das Feld zu erstellen. In diesem Fall wird eine leere dict erstellt, wenn kein Wert übergeben wird.
    